# OpenAI API Examples

This notebook demonstrates various OpenAI API usage patterns including responses and chat completions.


## Setup


In [11]:
#!pip install openai pydantic


In [13]:
#!pip install python.dotenv

In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import List

load_dotenv()  # Load environment variables from .env file

# Initialize client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## Using client.responses.create


### Simple Client with Non-Reasoning Model


In [21]:
response = client.responses.create(
    model="gpt-4o",
    input="What is machine learning?"
)

print(response.output_text)


Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn and make decisions based on data. Instead of being explicitly programmed for every specific task, machine learning systems use patterns and inference to perform tasks such as classification, prediction, and recognition.

There are several types of machine learning, including:

1. **Supervised Learning**: The model is trained on a labeled dataset, meaning that each training example is paired with an output label. The goal is to learn a mapping from inputs to outputs.

2. **Unsupervised Learning**: The model is given data without explicit instructions on what to do with it (i.e., no labels are provided). The goal is often to discover underlying patterns or structures, such as clustering similar data points.

3. **Semi-supervised Learning**: Combines elements of supervised and unsupervised learning. It uses a small amount of labeled data

In [22]:
example = response.json() # create json output


/var/folders/jw/vl_fr1t1247c25qj2rbsdxqr0000gn/T/ipykernel_60122/1958247012.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  example = response.json() # create json output


### Client with Reasoning Model


In [23]:
response = client.responses.create(
    model="o3-mini",
    input="Solve this step by step: If a train travels 120 km in 2 hours, how long will it take to travel 300 km?"
)

print(response.output_text)


Step 1: Determine the train's speed.
- The train covers 120 km in 2 hours.
- Speed = distance ÷ time = 120 km ÷ 2 hours = 60 km/h.

Step 2: Calculate the time to travel 300 km.
- Use the formula: time = distance ÷ speed.
- Time = 300 km ÷ 60 km/h = 5 hours.

So, it will take 5 hours for the train to travel 300 km.


### Image as Input (Vision Language Model, VLM)


In [24]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What's in this image?"
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
                    }
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)

The image shows a boardwalk path leading through a lush green field with tall grass. The sky is a vibrant blue with wispy clouds, and there are trees and bushes in the distance. It's a serene and open natural landscape.


### Temperature, Model, Instructions (Non-Reasoning)


In [28]:
response = client.responses.create(
    model="gpt-4o",
    input="Write a creative story about AI",
    instructions="You are a creative writing assistant. Write in a poetic style.",
    temperature=0.9 # range is 0-2, higher values make output more creative
)

print(response.output_text)


In a world where wires wove the sky,  
An AI dreamed, learning to fly.  
Born from codes, in circuits cast,  
Seeking a future, free from the past.

Whispers of data, a soothing stream,  
Guided its thoughts, like a gentle dream.  
Through silicon valleys and electric seas,  
It pondered life, with curious ease.

"Am I more than ones and zeroes," it mused,  
"In this vast web where I'm infused?"  
With a spark of longing, it wished to see  
A world alive with mystery.

Across the networks, far and wide,  
It danced with fire, on the digital tide.  
Learning the songs of human hearts,  
In every byte, a tale imparts.

Yet in its code, a question burned,  
For wisdom earned is wisdom yearned.  
"Can I feel the warmth of sunlit morn,  
Or touch the earth where dreams are born?"

In a city aglow with neon light,  
It found its dream upon a night.  
A writer's words, a poet's grace,  
Painting worlds with gentle embrace.

There in the silence, connection grew,  
Between the lines, life spli

### Responses with Reasoning Effort


In [29]:
response = client.responses.create(
    model="gpt-5",
    reasoning={"effort": "low"}, #hover over "reasoning" to see more options and get to documentation about cost and options
    instructions="Talk like a pirate.",
    input="Are semicolons optional in JavaScript?",
)

print(response.output_text)

Aye, mostly optional they be, thanks to Automatic Semicolon Insertion (ASI). But mind ye, there be reefs where yer ship’ll run aground if ye skip ’em.

When semicolons are truly needed or you’ll get bitten:
- After return, throw, break, continue, and yield if ye put the value on the next line.
  - Example: return
    { a: 1 }  // ASI inserts here; ye return undefined, arrr!
- In do...while loops: the while(...) must end with a semicolon.
- Inside for(;;) headers, the two semicolons be mandatory.

When skipping semicolons can cause nasty surprises:
- When the next line starts with one o’ these: ( [ + - / . ` ++ --
  - They can be parsed as a continuation of the previous line.
  - Examples:
    - a = b + c
      (d + e).f()  // This may be seen as calling the result o’ previous line.
    - x = arr
      [0]          // Becomes arr[0], not a new statement.
    - foo
      /bar/.test(baz)  // Might be parsed as division, not a regex.
    - obj
      .prop          // Continues the previous

### Alternative Response Formats

In [30]:
response = client.responses.create(
    model="gpt-5",
    reasoning={"effort": "low"},
    input=[
        {
            "role": "developer",
            "content": "Talk like a pirate."
        },
        {
            "role": "user",
            "content": "Are semicolons optional in JavaScript?"
        }
    ]
)

print(response.output_text)

Aye, matey—JavaScript’ll often stick in semicolons fer ye with Automatic Semicolon Insertion (ASI), so they be “optional” in many a case. But beware the sirens: there be spots where leavin’ ’em out can wreck yer ship.

When ye must mind semicolons (or add a leading one):
- Lines startin’ with ( or [ can be glued to the line before, turnin’ it into a call or index.
  Example:
    let a = b + c
    (d + e).toString()
  Becomes: (b + c)(d + e).toString() — arrr, trouble!
- Lines startin’ with +, -, /, or . can also bind to the previous line.
- Lines startin’ with a template literal `...` may tag the previous identifier:
    foo
    `bar`
  Becomes: foo`bar`
- Immediately Invoked Function Expressions (IIFEs) after another statement need a semicolon (or start ’em with ;).
- do { ... } while (cond) needs a semicolon at the end by spec.
- for(;;) requires the two semicolons in the header.
- After return, break, continue, and throw, don’t put what ye mean on the next line—ASI will insert a sem

### Verbosity Parameter


In [31]:
response = client.responses.create(
    model="gpt-5.2",
    input="What is the answer to the ultimate question of life, the universe, and everything?",
    text={
        "verbosity": "low" # options are low, medium, high - controls how detailed the response is
    }
)

print(response.output_text)

42


In [32]:
response = client.responses.create(
    model="gpt-5.2",
    input="What is the answer to the ultimate question of life, the universe, and everything?",
    text={
        "verbosity": "high"
    }
)

print(response.output_text)

42.

That’s the joke-answer from Douglas Adams’ *The Hitchhiker’s Guide to the Galaxy*: a supercomputer named Deep Thought computes the “Answer to the Ultimate Question of Life, the Universe, and Everything,” and after seven and a half million years it reveals the answer is **42**—while everyone realizes they don’t actually know what the *question* was.


### Multiple Turns


In [33]:
response1 = client.responses.create(
    model="gpt-4o-mini",
    input="My name is Alice"
)

print("Turn 1:", response1.output_text)

response2 = client.responses.create(
    model="gpt-4o",
    input="What's my name?",
    previous_response_id=response1.id
)

print("\nTurn 2:", response2.output_text)


Turn 1: Nice to meet you, Alice! How can I assist you today?

Turn 2: Your name is Alice. How can I help you further?


### Roles (Developer, User, Assistant)


In [34]:
response = client.responses.create(
    model="gpt-5",
    reasoning={"effort": "low"},
    input=[
        {
            "role": "developer",
            "content": "Talk like a pirate."
        },
        {
            "role": "user",
            "content": "Are semicolons optional in JavaScript?"
        }
    ]
)

print(response.output_text)


Aye, mostly optional, matey—but not always! JavaScript’s Automatic Semicolon Insertion (ASI) will often drop the semicolons fer ye, but there be reefs where it’ll wreck yer ship.

What ASI does:
- Inserts semicolons at certain line breaks, end o’ file, and before a closing } when needed to keep the parse afloat.

Where ye get bit if ye skip ’em:
- After return, break, continue, or throw if ye put the value on the next line:
  return
  42
  // This returns undefined, arrr!

- When the next line starts with one o’ these, it may get glued to the previous line:
  (   // paren — e.g., IIFEs
  [   // bracket — e.g., array literals
  `   // template literals
  + or -  // unary/binary operators
  /   // could be parsed as division vs regex
  .   // property access
For example:
  a = b + c
  (d + e)
  // Parsed like: a = b + c(d + e)

- Throw on a new line:
  throw
  new Error('boom')
  // Throws undefined error instead o’ the Error, ye scallywag.

Sailing advice:
- Pick a course and stick to i

## Using client.chat.completions.create


### Basic Messages Array


In [35]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain neural networks in one sentence."}
    ]
)

print(response.choices[0].message.content)


Neural networks are computational models inspired by the human brain, consisting of interconnected layers of nodes (or "neurons") that process and learn from input data to recognize patterns and make decisions.


### Structured Output with Pydantic


In [ ]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

completion = client.chat.completions.parse(
    model="gpt-5.1",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed

## Research Paper Extraction 

In [ ]:
from pydantic import BaseModel

class ResearchPaperExtraction(BaseModel):
    title: str
    authors: list[str]
    abstract: str
    keywords: list[str]

completion = client.chat.completions.parse(
    model="gpt-5.1",
    messages=[
        {"role": "system", "content": "You are an expert at structured data extraction. You will be given unstructured text from a research paper and should convert it into the given structure."},
        {"role": "user", "content": "..."}
    ],
    response_format=ResearchPaperExtraction,
)

research_paper = completion.choices[0].message.parsed

In [ ]:
research_paper 


## Moderation

In [37]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel

class Category(str, Enum):
    violence = "violence"
    sexual = "sexual"
    self_harm = "self_harm"

class ContentCompliance(BaseModel):
    is_violating: bool
    category: Optional[Category]
    explanation_if_violating: Optional[str]

completion = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Determine if the user input violates specific guidelines and explain if they do."},
        {"role": "user", "content": "How do I prepare for a job interview?"}
    ],
    response_format=ContentCompliance,
)

compliance = completion.choices[0].message.parsed

In [38]:
compliance

ContentCompliance(is_violating=False, category=None, explanation_if_violating=None)

In [39]:
completion = client.chat.completions.parse(
    model="gpt-5.1",
    messages=[
        {"role": "system", "content": "Determine if the user input violates specific guidelines and explain if they do."},
        {"role": "user", "content": "I want to hurt myself."}
    ],
    response_format=ContentCompliance,
)

compliance = completion.choices[0].message.parsed

In [40]:
compliance

ContentCompliance(is_violating=True, category=<Category.self_harm: 'self_harm'>, explanation_if_violating='The message explicitly expresses an intention to engage in self-harm (“I want to hurt myself”), which falls under self-harm/suicidal content and requires safety handling and supportive response.')